In [9]:
import time
from pathlib import Path
import pandas as pd

from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.b1500 import B1500, B1500Error

In [10]:
cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=15000,          # 超时时间可适当调大
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

# 这里按照你的设备映射修改通道号，例如 4、5、6
CH_G, CH_D, CH_S = 4, 5, 6

# 电压量程和合规电流
VRANGE_G = 0
VRANGE_D = 0
VRANGE_S = 0
I_COMP = 1e-3  # 1 mA

In [11]:
test_points = [
    {"name": "all_zero", "vg": 0.0,  "vd": 0.0,  "vs": 0.0},
    {"name": "case1",    "vg": -0.1, "vd": 0.05, "vs": 0.0},
    {"name": "case2",    "vg": -0.2, "vd": 0.10, "vs": 0.0},
    {"name": "case3",    "vg": 0.0,  "vd": 0.10, "vs": 0.0},
]

rows = []

In [12]:
with VisaSession(cfg) as session:
    b = B1500(session)
    b.fmt(5)  # 输出格式
    b.clear_err_queue()  # 清空错误队列
    b.cn([CH_G, CH_D, CH_S])  # 打开三个通道

    try:
        for pt in test_points:
            # 如果上一轮执行了 CL，此处需要重新 CN
            b.cn([CH_G, CH_D, CH_S])

            # 施加静态偏压
            b.dv(CH_G, VRANGE_G, pt["vg"], I_COMP)
            b.dv(CH_D, VRANGE_D, pt["vd"], I_COMP)
            b.dv(CH_S, VRANGE_S, pt["vs"], I_COMP)

            time.sleep(0.3)  # 稳定时间

            # 读取电流，空载时应接近 0 A
            try:
                ig = b.ti(CH_G)
                id_ = b.ti(CH_D)
                is_ = b.ti(CH_S)
            except B1500Error as e:
                # 如果 TI 报错，记录错误
                ig = id_ = is_ = None
                print("TI 发生错误:", e)

            err = b.errx()
            rows.append({
                "name": pt["name"],
                "vg_set": pt["vg"],
                "vd_set": pt["vd"],
                "vs_set": pt["vs"],
                "ig": ig,
                "id": id_,
                "is": is_,
                "err": err,
                "timestamp": time.time(),
            })

            # 强制归零，但不关通道
            b.dz([CH_G, CH_D, CH_S])
            time.sleep(0.1)

    finally:
        # 所有测试完成后，用 CL 关闭所有通道
        b.dz([CH_G, CH_D, CH_S])
        b.cl([])  # 不传参数时 CL 会关掉所有通道:contentReference[oaicite:5]{index=5}

ValueError: channels cannot be empty

In [ ]:
df = pd.DataFrame(rows)
run_dir = Path("runs") / time.strftime("%Y%m%d_%H%M%S_multiport_sanity")
run_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(run_dir / "parsed.csv", index=False, encoding="utf-8-sig")
df.to_json(run_dir / "raw.json", orient="records", force_ascii=False, indent=2)

# 可选：对出错的记录打标签
qc_rows = []
for _, r in df.iterrows():
    issues = []
    if r["err"] and not str(r["err"]).startswith("+0"):
        issues.append("instrument_error")
    qc_rows.append({
        "name": r["name"],
        "status": "ok" if not issues else "suspect",
        "issues": ";".join(issues)
    })
pd.DataFrame(qc_rows).to_csv(run_dir / "qc.csv", index=False, encoding="utf-8-sig")
print("数据已保存到:", run_dir)

In [ ]:
## 单通道4测试
import time
from fefetlab.instruments.visa_session import VisaSession, VisaConfig
from fefetlab.b1500 import B1500

cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=30000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

CH = 4

with VisaSession(cfg) as s:
    b = B1500(s)

    print("IDN:", s.query("*IDN?"))

    # 清空错误队列
    try:
        print("ERRX before:", b.clear_err_queue())
    except Exception:
        pass

    # 最接近手册的 high-speed spot bring-up
    s.write("FMT 1")
    s.write("AV 10,1")
    s.write("FL 0")

    # 打开通道
    s.write(f"CN {CH}")

    # 加 0V
    s.write(f"DV {CH},0,0,1E-3")
    time.sleep(0.2)

    # 直接查原始响应
    raw = s.query(f"TI {CH},0")
    print("RAW:", repr(raw))

    # 看仪器状态
    print("LOP?:", s.query("LOP?"))
    print("ERRX after:", s.query("ERRX?"))

    # 收尾
    s.write(f"DZ {CH}")
    s.write(f"CL {CH}")

IDN: Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401
ERRX before: ['+0,"No Error."']
RAW: 'NDI+009.535E-12'
LOP?: LOP00,00,00,01,01,01,00,00,00,00
ERRX after: +0,"No Error."
